# 03 — Pipeline CNBI e comparações

Implementa payoff, orientação canônica, análise paralela legada independente, janela singular, deltas adaptativos e subproblemas CNBI. O SMOKE valida o núcleo; PILOT/FULL habilitam os métodos completos sob contador real.

In [ ]:
from pathlib import Path
import json, os, time, math, gc
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.spatial import ConvexHull, Delaunay, cKDTree
from scipy.stats import qmc

def project_root(start=Path.cwd()):
    p=start.resolve()
    for candidate in (p,*p.parents):
        if (candidate/'configs'/'smoke.json').exists(): return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada')

ROOT=project_root(); MODE=os.environ.get('CNBI_MODE','SMOKE').upper()
if MODE=='FULL' and os.environ.get('CNBI_FULL_CONFIRMED')!='YES':
    raise RuntimeError('FULL bloqueado: defina CNBI_FULL_CONFIRMED=YES após autorização explícita.')
CFG=json.loads((ROOT/'configs'/f'{MODE.lower()}.json').read_text(encoding='utf-8'))
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[key]='1'
ALPHA=2**0.75; DELTA_BY_K={2:.10,3:.10,4:.20,5:.50}

from itertools import combinations,product
from scipy.linalg import null_space
OUT=ROOT/'data'/'generated'; OUT=OUT/'smoke' if MODE=='SMOKE' else OUT; CK=ROOT/'results'/'checkpoints'; TAB=ROOT/'results'/'tables'; CK.mkdir(parents=True,exist_ok=True); TAB.mkdir(parents=True,exist_ok=True)

class EvaluationCounter:
    def __init__(self,predict): self.predict,self.full,self.grad=predict,0,0
    def __call__(self,x): self.full+=1; return np.asarray(self.predict(np.asarray(x,float).reshape(1,-1))[0],float)

def z(x):
    x1,x2,x3=np.asarray(x,float); return np.array([1,x1,x2,x3,x1*x1,x2*x2,x3*x3,x1*x2,x1*x3,x2*x3])
def dz(x):
    x1,x2,x3=np.asarray(x,float); return np.array([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[2*x1,0,0],[0,2*x2,0],[0,0,2*x3],[x2,x1,0],[x3,0,x1],[0,x3,x2]])

def individual_payoff(B):
    m=B.shape[1]; Xs=[]; cols=[]; evaluation_count=0; gradient_count=0
    for j in range(m):
        best=None
        starts=[np.zeros(3),*list(np.eye(3)*(.99*ALPHA)),*list(-np.eye(3)*(.99*ALPHA))]
        for x0 in starts:
            def objective(x):
                nonlocal evaluation_count; evaluation_count+=1; return float(z(x)@B[:,j])
            def gradient(x):
                nonlocal gradient_count; gradient_count+=1; return dz(x).T@B[:,j]
            r=minimize(objective,x0,jac=gradient,method='SLSQP',bounds=[(-ALPHA,ALPHA)]*3,constraints={'type':'ineq','fun':lambda x:ALPHA**2-x@x},options={'ftol':1e-11,'maxiter':500})
            if best is None or r.fun<best.fun: best=r
        assert best.success and best.x@best.x<=ALPHA**2+1e-7
        Xs.append(best.x); cols.append(z(best.x)@B)
    # Public orientation: rows objectives, columns individual configurations.
    P=np.column_stack(cols)
    individual_payoff.last_evaluations=evaluation_count; individual_payoff.last_gradient_evaluations=gradient_count
    return np.asarray(Xs),P

def parallel_analysis(P,Xstar,mse,XtX_inv,nmc=2000,seed=777):
    ideal=P.min(1); nadir=P.max(1); amp=np.where(nadir-ideal>1e-12,nadir-ideal,1); Ps=(P-ideal[:,None])/amp[:,None]
    # v0 adapter: internal anchors-in-rows equals public payoff transpose.
    A=Ps.T; assert np.allclose(A.T,Ps)
    E=A[1:]-A[:1]; s=np.linalg.svd(E,compute_uv=False); h=np.einsum('ij,jk,ik->i',np.vstack([z(x) for x in Xstar]),XtX_inv,np.vstack([z(x) for x in Xstar]))
    sd=np.sqrt(np.outer(h,mse))/amp[None,:]; rng=np.random.default_rng(seed); sn=np.empty((nmc,len(s)))
    for b in range(nmc):
        R=rng.normal(0,sd); sn[b]=np.linalg.svd(R[1:]-R[:1],compute_uv=False)
    p95=np.percentile(sn,95,axis=0); d=max(1,int(np.sum(s>p95))); floor=float(s[d]) if d<len(s) else 0.; ceiling=float(s[0]/s[d-1])
    return {'d':d,'s':s,'p95':p95,'floor':floor,'ceiling':ceiling,'scaled':Ps}

def simplex_weights(k,delta):
    p=round(1/delta); return np.array([q for q in product(range(p+1),repeat=k) if sum(q)==p],float)/p

def _nearest_weight_order(weights):
    weights=np.asarray(weights,float); remaining=list(range(len(weights))); order=[remaining.pop(0)]
    while remaining:
        last=weights[order[-1]]; pick=min(remaining,key=lambda i:(float(np.linalg.norm(weights[i]-last)),i)); remaining.remove(pick); order.append(pick)
    return [(i,weights[i]) for i in order]

def _nearest_weight_order(weights):
    weights=np.asarray(weights,float); remaining=list(range(len(weights))); order=[remaining.pop(0)]
    while remaining:
        last=weights[order[-1]]; pick=min(remaining,key=lambda i:(float(np.linalg.norm(weights[i]-last)),i)); remaining.remove(pick); order.append(pick)
    return [(i,weights[i]) for i in order]

def cnbi(B,P,Xstar,mse,XtX_inv):
    """CNBI v0: anchor-safe vertices, analytic Jacobians, warm starts and deterministic rescue."""
    pa=parallel_analysis(P,Xstar,mse,XtX_inv,nmc=2000,seed=777); m=B.shape[1]; results=[]; evaluation_count=0; gradient_count=0
    for k in range(2,min(m,4)+1):
      for combo_id,combo in enumerate(combinations(range(m),k)):
        combo=np.asarray(combo,int); A=pa['scaled'][np.ix_(combo,combo)].T; E=A[1:]-A[:1]; s=np.linalg.svd(E,compute_uv=False); q=np.inf if s[-1]<=1e-12 else s[0]/s[-1]
        if not(s[-1]>pa['floor'] and q<=pa['ceiling']): continue
        normal=null_space(E).ravel(); normal/=np.linalg.norm(normal)
        if normal@(-A.mean(0))<0: normal=-normal
        anchors=Xstar[combo]; ideal=P[combo].min(1); amp=np.maximum(P[combo].max(1)-ideal,1e-12); warm=None
        for beta_id,beta in _nearest_weight_order(simplex_weights(k,DELTA_BY_K[k])):
            phi=beta@A
            vertex=np.flatnonzero(np.isclose(beta,1.,atol=1e-12)&np.isclose(beta.sum(),1.,atol=1e-12))
            if len(vertex)==1:
                x=anchors[int(vertex[0])].copy(); Fv=z(x)@B; evaluation_count+=1; eq=(Fv[combo]-ideal)/amp-phi
                results.append({'k':k,'combo':tuple(combo),'beta_id':beta_id,'beta':beta.copy(),'success':bool(np.max(np.abs(eq))<=1e-5),'x':x,'F_rsm':Fv,'eq_inf':float(np.max(np.abs(eq))),'sphere_violation':max(0.,float(x@x-ALPHA**2)),'start':'payoff_anchor_exact','attempts':0})
                warm=np.r_[x,0.]; continue
            def predict_scaled(x):
                nonlocal evaluation_count; evaluation_count+=1; return (z(x)@B[:,combo]-ideal)/amp
            def predict_jac(x):
                nonlocal gradient_count; gradient_count+=1; return (dz(x).T@B[:,combo]).T/amp[:,None]
            def eq(v): return predict_scaled(v[:3])-(phi+v[-1]*normal)
            def jeq(v): return np.column_stack([predict_jac(v[:3]),-normal])
            def sphere(v): return ALPHA**2-v[:3]@v[:3]
            def jsphere(v): return np.r_[-2*v[:3],0.]
            def project_t(x):
                nonlocal evaluation_count; evaluation_count+=1; return float(((z(x)@B[:,combo]-ideal)/amp-phi)@normal)
            starts=[]; seen=set()
            def add(name,x,t=None):
                x=np.asarray(x,float); norm=np.linalg.norm(x)
                if norm>ALPHA*(1+1e-10): return
                if norm>ALPHA: x=x*(ALPHA*(1-1e-12)/norm)
                tt=project_t(x) if t is None else float(t); v=np.r_[x,np.clip(tt,-10,10)]; key=tuple(np.round(v,12))
                if key not in seen: seen.add(key); starts.append((name,v))
            if warm is not None: add('warm',warm[:3],warm[-1])
            add('barycentric_anchor',beta@anchors)
            for j in np.argsort(-beta): add(f'anchor_{int(combo[j])}',anchors[j])
            add('center',np.zeros(3))
            candidates=[]
            def solve(name,v0,attempt):
                r=minimize(lambda v:-v[-1],v0,jac=lambda v:np.r_[np.zeros(3),-1.],method='SLSQP',bounds=[(-ALPHA,ALPHA)]*3+[(-10,10)],constraints=[{'type':'eq','fun':eq,'jac':jeq},{'type':'ineq','fun':sphere,'jac':jsphere}],options={'ftol':1e-10,'maxiter':500,'disp':False})
                ev=eq(r.x); eq_inf=float(np.max(np.abs(ev))); sv=max(0.,float(r.x[:3]@r.x[:3]-ALPHA**2)); feasible=eq_inf<=1e-5 and sv<=1e-8; cand=(bool(r.success),feasible,r,name,attempt,eq_inf,sv); candidates.append(cand); return cand
            converged=False; attempt=0
            for name,v0 in starts:
                attempt+=1; cand=solve(name,v0,attempt)
                if cand[0] and cand[1]: converged=True; break
            if not converged:
                rng=np.random.default_rng(1000003+1009*combo_id+beta_id)
                for rescue in range(8):
                    direction=rng.normal(size=3); direction/=np.linalg.norm(direction); x=ALPHA*(rng.random()**(1/3))*direction; attempt+=1; cand=solve(f'rescue_{rescue+1}',np.r_[x,np.clip(project_t(x),-10,10)],attempt)
                    if cand[0] and cand[1]: break
            chosen=min(candidates,key=lambda c:(0 if c[1] else 1,-float(c[2].x[-1]) if c[1] else c[5]/1e-5+c[6]/1e-8,c[5],c[6])); success,feasible,r,name,attempt,eq_inf,sv=chosen; x=r.x[:3]; Fv=z(x)@B; evaluation_count+=1
            results.append({'k':k,'combo':tuple(combo),'beta_id':beta_id,'beta':beta.copy(),'success':bool(success and feasible),'x':x,'F_rsm':Fv,'eq_inf':eq_inf,'sphere_violation':sv,'start':name,'attempts':attempt})
            warm=r.x.copy() if feasible else None
    pa['rsm_evaluations']=evaluation_count; pa['gradient_evaluations']=gradient_count
    return results,pa

# SMOKE checks the legacy core without running expensive optimizer comparisons.
diag=pd.read_csv(OUT/'scenario_diagnostics.csv'); rec=diag.iloc[0]; A=np.load(OUT/f"{rec.scenario}_scenario.npz")['anchors']
X=np.vstack([np.array(list(product([-1.,1.],repeat=3))),np.vstack([np.eye(3)*ALPHA,-np.eye(3)*ALPHA]),np.zeros((5,3))]); D=np.vstack([z(x) for x in X]); F=np.sum((X[:,None,:]-A[None,:,:])**2,axis=2)
rng=np.random.default_rng(101); sig=np.sqrt(F.var(0,ddof=1)*(.05/.95)); Y=F+rng.normal(0,sig,F.shape); B=np.linalg.lstsq(D,Y,rcond=None)[0]; E=Y-D@B; mse=np.sum(E*E,axis=0)/(19-10)
Xs,P=individual_payoff(B); pa=parallel_analysis(P,Xs,mse,np.linalg.inv(D.T@D),nmc=2000,seed=777)
raw_smoke,pa_cnbi_smoke=cnbi(B,P,Xs,mse,np.linalg.inv(D.T@D)); assert raw_smoke and all(r['sphere_violation']<=1e-8 for r in raw_smoke); assert any(r['start']=='payoff_anchor_exact' for r in raw_smoke); assert pa_cnbi_smoke['rsm_evaluations']>0 and pa_cnbi_smoke['gradient_evaluations']>0
assert pa['d']>=1 and np.all(np.isfinite(pa['p95'])) and DELTA_BY_K=={2:.1,3:.1,4:.2,5:.5}
pd.DataFrame([{'scenario':rec.scenario,'parallel_d':pa['d'],'svd_calls':2001,'rsm_evaluations_payoff':'instrumented_in_FULL'}]).to_csv(TAB/'smoke_pipeline.csv',index=False)
print('SMOKE pipeline OK; otimização final executada?',CFG['run_optimizers'])


## Execução permanente dos métodos determinísticos

Executa NBI direto, CNBI e VRF-NBI sobre o mesmo RSM pareado e grava checkpoints por método, cenário e semente.

In [ ]:
# PERMANENT PILOT METHOD EXECUTION
# This cell executes deterministic methods and creates one checkpoint per method/scenario/seed.
from factor_analyzer import FactorAnalyzer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def _normalized_model(B, rows):
    Xs,P=individual_payoff(B[:,rows])
    ideal=P.min(1); amp=np.maximum(P.max(1)-ideal,1e-12)
    return Xs,P,ideal,amp

def direct_nbi(B, delta=.20):
    m=B.shape[1]
    if m!=4:
        return [], {'status':'STRUCTURALLY_INVALID','reason':'m > n_x + 1'}
    Xs,P,ideal,amp=_normalized_model(B,np.arange(m)); payoff_evals=individual_payoff.last_evaluations; payoff_grads=individual_payoff.last_gradient_evaluations; A=((P-ideal[:,None])/amp[:,None]).T
    E=A[1:]-A[:1]; normal=null_space(E).ravel(); normal/=np.linalg.norm(normal)
    if normal@(-A.mean(0))<0: normal=-normal
    rows=[]; counter=0
    for beta in simplex_weights(m,delta):
        phi=beta@A; x0=beta@Xs
        def eq(v):
            nonlocal counter; counter+=1
            return (z(v[:3])@B-ideal)/amp-(phi+v[-1]*normal)
        r=minimize(lambda v:-v[-1],np.r_[x0,0.],method='SLSQP',bounds=[(-ALPHA,ALPHA)]*3+[(-10,10)],constraints=[{'type':'eq','fun':eq},{'type':'ineq','fun':lambda v:ALPHA**2-v[:3]@v[:3]}],options={'ftol':1e-9,'maxiter':500})
        rows.append({'x':r.x[:3],'F_rsm':z(r.x[:3])@B,'success':bool(r.success and np.max(np.abs(eq(r.x)))<=1e-5)})
    return rows, {'status':'COMPLETED','rsm_evaluations':counter+payoff_evals,'payoff_evaluations':payoff_evals,'gradient_evaluations':payoff_grads,'payoff_included':True}

def vrf_nbi(B,Yobs,delta=.10):
    Ys=StandardScaler().fit_transform(Yobs); pca=PCA().fit(Ys); k=int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_),.90)+1)
    if k>4:
        return [], {'status':'STRUCTURALLY_INVALID','n_factors':k,'reason':'retained factors > n_x + 1'}
    fa=FactorAnalyzer(n_factors=k,rotation='varimax',method='principal'); scores=fa.fit_transform(Ys)
    Bf=np.linalg.lstsq(D,scores,rcond=None)[0]
    Xs,P,ideal,amp=_normalized_model(Bf,np.arange(k)); payoff_evals=individual_payoff.last_evaluations; payoff_grads=individual_payoff.last_gradient_evaluations; A=((P-ideal[:,None])/amp[:,None]).T
    E=A[1:]-A[:1]
    if k==1: return [], {'status':'STRUCTURALLY_INVALID','n_factors':k,'reason':'NBI requires at least two factors'}
    normal=null_space(E).ravel(); normal/=np.linalg.norm(normal)
    if normal@(-A.mean(0))<0: normal=-normal
    rows=[]; counter=0
    for beta in simplex_weights(k,delta):
        phi=beta@A; x0=beta@Xs
        def eq(v):
            nonlocal counter; counter+=1
            return (z(v[:3])@Bf-ideal)/amp-(phi+v[-1]*normal)
        r=minimize(lambda v:-v[-1],np.r_[x0,0.],method='SLSQP',bounds=[(-ALPHA,ALPHA)]*3+[(-10,10)],constraints=[{'type':'eq','fun':eq},{'type':'ineq','fun':lambda v:ALPHA**2-v[:3]@v[:3]}],options={'ftol':1e-9,'maxiter':500})
        rows.append({'x':r.x[:3],'F_rsm':z(r.x[:3])@B,'success':bool(r.success and np.max(np.abs(eq(r.x)))<=1e-5)})
    return rows, {'status':'COMPLETED','n_factors':k,'rsm_evaluations':counter+payoff_evals,'payoff_evaluations':payoff_evals,'gradient_evaluations':payoff_grads,'payoff_included':True,'loadings':fa.loadings_.tolist()}

def save_front(path,rows,meta):
    path.parent.mkdir(parents=True,exist_ok=True)
    Xout=np.vstack([r['x'] for r in rows]) if rows else np.empty((0,3)); Fout=np.vstack([r['F_rsm'] for r in rows]) if rows else np.empty((0,0))
    extras={}
    if rows and 'eq_inf' in rows[0]:
        extras={'k':np.array([r['k'] for r in rows],int),'beta_id':np.array([r['beta_id'] for r in rows],int),'combo_json':np.array([json.dumps(np.asarray(r['combo'],int).tolist()) for r in rows]),'beta_json':np.array([json.dumps(np.asarray(r['beta']).tolist()) for r in rows]),'eq_inf':np.array([r['eq_inf'] for r in rows],float),'sphere_violation':np.array([r['sphere_violation'] for r in rows],float),'start':np.array([r['start'] for r in rows]),'attempts':np.array([r['attempts'] for r in rows],int)}
    np.savez_compressed(path,X=Xout,F_rsm=Fout,success=np.array([r['success'] for r in rows],bool),metadata=json.dumps(meta),**extras)

def execute_deterministic_campaign():
    if not CFG['run_optimizers']: return pd.DataFrame()
    manifest=[]; diag=pd.read_csv(OUT/'scenario_diagnostics.csv')
    for scenario in diag.scenario:
      if scenario not in [f'm{m}_{level}' for m in CFG['scenario_objectives'] for level in CFG['correlation_targets']]: continue
      Atrue=np.load(OUT/f'{scenario}_scenario.npz')['anchors']; m=len(Atrue)
      for seed in CFG['final_seeds']:
        Ftrue=np.sum((X[:,None,:]-Atrue[None,:,:])**2,axis=2); rng=np.random.default_rng(seed); sigma=np.sqrt(Ftrue.var(0,ddof=1)*(.05/.95)); Y=Ftrue+rng.normal(0,sigma,Ftrue.shape)
        B=np.linalg.lstsq(D,Y,rcond=None)[0]; E=Y-D@B; mse=np.sum(E*E,axis=0)/9; Xs,P=individual_payoff(B); payoff_evals=individual_payoff.last_evaluations; payoff_grads=individual_payoff.last_gradient_evaluations
        for method in ['NBI','CNBI','VRF-NBI']:
          ck=CK/f'{MODE.lower()}_{scenario}_seed{seed}_{method.replace("/","_")}_v0safe_audit.npz'; t0=time.perf_counter(); c0=time.process_time()
          if ck.exists():
            dat=np.load(ck,allow_pickle=False); meta=json.loads(str(dat['metadata'])); status=meta['status']; n=len(dat['X'])
            if method=='CNBI' and not meta.get('payoff_included',False):
                meta={**meta,'cnbi_subproblem_evaluations':int(meta['rsm_evaluations']),'payoff_evaluations':int(payoff_evals),'gradient_evaluations':int(payoff_grads),'rsm_evaluations':int(meta['rsm_evaluations'])+int(payoff_evals),'payoff_included':True}
                np.savez_compressed(ck,X=dat['X'],F_rsm=dat['F_rsm'],success=dat['success'],metadata=json.dumps(meta))
          else:
            if method=='NBI': rows,meta=direct_nbi(B)
            elif method=='VRF-NBI': rows,meta=vrf_nbi(B,Y)
            else:
                raw,pa=cnbi(B,P,Xs,mse,np.linalg.inv(D.T@D)); rows=raw; meta={'status':'COMPLETED','parallel_d':pa['d'],'cnbi_subproblem_evaluations':pa['rsm_evaluations'],'payoff_evaluations':payoff_evals,'gradient_evaluations':pa.get('gradient_evaluations',0)+payoff_grads,'rsm_evaluations':pa['rsm_evaluations']+payoff_evals,'payoff_included':True}
            save_front(ck,rows,meta); status=meta['status']; n=len(rows)
          manifest.append({'scenario':scenario,'seed':seed,'method':method,'status':status,'n_solutions':n,'rsm_evaluations':meta.get('rsm_evaluations',0),'gradient_evaluations':meta.get('gradient_evaluations',0),'wall_seconds':time.perf_counter()-t0,'cpu_seconds':time.process_time()-c0,'checkpoint':str(ck.relative_to(ROOT))})
    out=pd.DataFrame(manifest); out.to_csv(TAB/f'{MODE.lower()}_deterministic_runs.csv',index=False); return out

deterministic_manifest=execute_deterministic_campaign()
if CFG['run_optimizers']: print(deterministic_manifest.to_string(index=False))


## Artefatos diagnósticos auditáveis

Exporta RSM, payoff, análise paralela, VRF e o ledger combinação–peso do CNBI, além dos checkpoints agregados retomáveis.

In [ ]:
# SCIENTIFIC DIAGNOSTIC ARTIFACTS
def export_scientific_diagnostics():
    if not CFG['run_optimizers']: return
    rsm_rows=[]; payoff_rows=[]; pa_rows=[]; vrf_rows=[]; ledger=[]; diag=pd.read_csv(OUT/'scenario_diagnostics.csv')
    wanted={f'm{m}_{level}' for m in CFG['scenario_objectives'] for level in CFG['correlation_targets']}
    for scenario in diag[diag.scenario.isin(wanted)].scenario:
      Atrue=np.load(OUT/f'{scenario}_scenario.npz')['anchors']; m=len(Atrue)
      for seed in CFG['final_seeds']:
        Ftrue=np.sum((X[:,None,:]-Atrue[None,:,:])**2,axis=2); rng=np.random.default_rng(seed); sigma=np.sqrt(Ftrue.var(0,ddof=1)*(.05/.95)); Y=Ftrue+rng.normal(0,sigma,Ftrue.shape); B=np.linalg.lstsq(D,Y,rcond=None)[0]; fitted=D@B; E=Y-fitted; mse=np.sum(E*E,axis=0)/9; B0=np.linalg.lstsq(D,Ftrue,rcond=None)[0]
        for j in range(m):
          ssr=float(np.sum((Y[:,j]-fitted[:,j])**2)); sst=float(np.sum((Y[:,j]-Y[:,j].mean())**2)); signal=float(np.var(Ftrue[:,j],ddof=1)); noise=float(np.var(Y[:,j]-Ftrue[:,j],ddof=1)); rsm_rows.append({'scenario':scenario,'seed':seed,'objective':j,'n_design':len(D),'design_rank':int(np.linalg.matrix_rank(D)),'r2_target':.95,'r2_observed':1-ssr/sst,'mse_residual':mse[j],'snr_realized':signal/max(noise,1e-15),'noiseless_max_abs_error':float(np.max(np.abs(D@B0[:,j]-Ftrue[:,j])))})
        Xs,P=individual_payoff(B); payoff_rows.append({'scenario':scenario,'seed':seed,'rows':P.shape[0],'columns':P.shape[1],'orientation':'rows_objectives_columns_optima','transpose_equivalent':bool(np.allclose(P.T.T,P)),'all_optima_feasible':bool(np.all(np.sum(Xs*Xs,axis=1)<=ALPHA**2+1e-8))})
        pa=parallel_analysis(P,Xs,mse,np.linalg.inv(D.T@D),nmc=2000,seed=777)
        for idx,(sv,p95) in enumerate(zip(pa['s'],pa['p95'])): pa_rows.append({'scenario':scenario,'seed':seed,'singular_index':idx+1,'singular_value':sv,'noise_p95':p95,'retained_d':pa['d'],'n_mc':2000,'mc_seed':777,'mode':'independent_legacy','floor':pa['floor'],'ceiling':pa['ceiling']})
        Ys=StandardScaler().fit_transform(Y); pca=PCA().fit(Ys); k=int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_),.90)+1); fa=FactorAnalyzer(n_factors=k,rotation='varimax',method='principal'); scores=fa.fit_transform(Ys)
        vrf_rows.append({'scenario':scenario,'seed':seed,'n_factors':k,'pca_cumulative':float(np.cumsum(pca.explained_variance_ratio_)[k-1]),'factor_method':'principal','rotation':'varimax','loadings_json':json.dumps(fa.loadings_.tolist()),'scores_shape':json.dumps(list(scores.shape)),'structurally_valid':bool(2<=k<=4)})
        ck=CK/f'{MODE.lower()}_{scenario}_seed{seed}_CNBI_v0safe_audit.npz'; dat=np.load(ck,allow_pickle=False); successes=np.asarray(dat['success'],bool); cursor=0
        for kk in range(2,min(m,4)+1):
          for combo in combinations(range(m),kk):
            AA=pa['scaled'][np.ix_(combo,combo)].T; ss=np.linalg.svd(AA[1:]-AA[:1],compute_uv=False); quality=np.inf if ss[-1]<=1e-12 else ss[0]/ss[-1]
            if not(ss[-1]>pa['floor'] and quality<=pa['ceiling']): continue
            weights=simplex_weights(kk,DELTA_BY_K[kk])
            for bi,beta in enumerate(weights):
              ledger.append({'scenario':scenario,'seed':seed,'k':kk,'combo':json.dumps(combo),'beta_index':bi,'beta':json.dumps(beta.tolist()),'delta':DELTA_BY_K[kk],'success':bool(successes[cursor]),'eq_inf':float(dat['eq_inf'][cursor]),'sphere_violation':float(dat['sphere_violation'][cursor]),'start':str(dat['start'][cursor]),'attempts':int(dat['attempts'][cursor]),'aggregate_checkpoint':str(ck.relative_to(ROOT))}); cursor+=1
        assert cursor==len(successes),f'Ledger CNBI incompatível: {scenario}, seed {seed}, {cursor}!={len(successes)}'
    pd.DataFrame(rsm_rows).to_csv(TAB/f'{MODE.lower()}_rsm_diagnostics.csv',index=False); pd.DataFrame(payoff_rows).to_csv(TAB/f'{MODE.lower()}_payoff_diagnostics.csv',index=False); pd.DataFrame(pa_rows).to_csv(TAB/f'{MODE.lower()}_parallel_analysis.csv',index=False); pd.DataFrame(vrf_rows).to_csv(TAB/f'{MODE.lower()}_vrf_diagnostics.csv',index=False); pd.DataFrame(ledger).to_csv(TAB/f'{MODE.lower()}_cnbi_subproblems.csv',index=False)

export_scientific_diagnostics()